# [서브] 3DAR_EV FaultBandNet 3×1 — 위치별 고장 여부만 판정

주 노트북(`../3DAR_EV_FaultBandNet.ipynb`, 3×3)과 입력·신경망·분할·학습이 같고 **마지막 층만 2304 → 3**(위치별 고장 여부)이다. 라벨 = 그 스피커가 Whine/PWM/Wind면 1. 검증 지표는 섹터 AUC. 비교용 서브 실험.

전기차 고장음을 3DAR(회전 메타구조 마이크)로 녹음한 뒤, **고장이 나타나는 대역만** 각도별로 뽑아 신경망에 넣는다.

- 입력: 녹음 1개(한 바퀴) → 지도 3장 (마이크 각도 36칸 × 주파수), 모두 녹음 안 상대값
  - Wind 3.3–6.2 kHz (36×58) · PWM 4.8–5.2 kHz (36×68) · Whine order 16 ± 150 Hz (36×51)
  - 각도는 360° 전부, 주파수는 0–24 kHz 중 약 13 %만 사용
- 모델: 지도별 2D CNN → 각도 방향 1D CNN(원형) → FC → **3×1 = 위치 3(앞/뒤/오른쪽) × 고장 여부 확률**
- 데이터: 3스피커 녹음 1,200개 중 **Imbalance·Modulation이 하나라도 들어간 녹음 제외 → 664개**
  (두 고장은 녹음에서 성분이 사라져 녹음 조건만 배움)
- 평가: **원본 음원 비중복 반복 분할 30회**. 매번 원본 파일 90개를 시험용으로 떼고,
  시험 = 세 스피커 원본이 모두 그 안에 있는 녹음, 학습 = 그 파일이 하나도 없는 녹음.
  어떤 스피커의 원본 파일도 학습과 시험에 겹치지 않는다. 녹음별로 시험된 회차의 확률을 평균.
  (이전 버전의 5-fold는 오른쪽 스피커만 묶어서 시험 녹음의 98 %가 앞·뒤 원본을 학습과 공유했음)

셀 구성(펭귄 노트북과 같은 순서): **셀 1** 설정·데이터·특징·모델·학습 → **셀 2** 평가·요약 → **셀 3–4** 그림.
데이터(측정 WAV)는 저장소에 포함하지 않는다. `CONFIG['data_root']`만 바꾸면 된다.


In [ ]:
# ==========================================
# 1. Configuration
# ==========================================
import csv, json, time, random, hashlib, os
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import soundfile as sf
from scipy import signal
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
import torch
from torch import nn
import torch.nn.functional as F

CONFIG = {
    # --- 경로 ---
    'data_root': '/home/kos/Desktop/3DAR_EV/[3DAR Electric Car]/3DAR_EV',   # 측정_목록.csv 가 있는 폴더 (읽기 전용)
    'out_dir': '../runs/nb_faultbandnet_strict_20261124',                     # 이 노트북의 모든 결과
    'feature_cache_dir': '../runs/nb_faultbandnet_20261124',                  # 특징 캐시(녹음 1,200개) 재사용
    'device': 'cuda', 'train': True, 'workers': 10,
    # --- 녹음 / 위치 ---
    'sample_rate': 48000, 'positions': ['FRONT', 'REAR', 'RIGHT'], 'fault_types': ['Whine', 'PWM', 'Wind'],
    # --- 특징: 고장 대역 × 마이크 각도 지도 ---
    'n_fft': 8192, 'hop_length': 480, 'angle_bins': 36,
    'wind_band': [3300, 6200], 'wind_bin_hz': 50, 'wind_flanks': [4000, 5500],     # 양옆(<4.0, >5.5 kHz)에 맞춘 직선을 뺌
    'pwm_band': [4800, 5200], 'pwm_bins': 68, 'pwm_line_exclude': [4975, 5025],    # Wind 지도에서는 5 kHz 선을 제외
    'whine_order': 16, 'whine_halfwidth_bins': 25,                                  # order 16 주파수 ± 25칸(≈150 Hz)
    'rpm_refine_orders': [4, 8, 12, 16],                                            # 정상 order가 가장 선명해지도록 추정 RPM 보정
    # --- 학습 ---
    'exclude_classes': ['Imbalance', 'Modulation'],                                 # 하나라도 들어간 녹음 제외
    'output_mode': '3x1',                                                         # 3x3: 위치×고장종류, 3x1: 위치×고장여부
    'repeats': 30, 'test_files_per_repeat': 90, 'split_seed': 0,                    # 원본 음원 비중복 반복 분할
    'validation_fraction': 0.15, 'seed_base': 1000,
    'batch_size': 32, 'epochs': 80, 'patience': 15, 'learning_rate': 1e-3, 'weight_decay': 1e-3,
    'dropout': 0.3, 'max_pos_weight': 20.0, 'threshold': 0.5, 'gpu_duty': 0.5,
}
POS, TY = CONFIG['positions'], CONFIG['fault_types']
OUT = Path(CONFIG['out_dir']); OUT.mkdir(parents=True, exist_ok=True)

# ==========================================
# 0. 보조 함수
# ==========================================
def read_csv(path):
    with open(path, encoding='utf-8-sig', newline='') as f: return list(csv.DictReader(f))

def write_csv(path, rows):
    with open(path, 'w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

def write_json(path, obj): Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding='utf-8')

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

class GpuThrottle:
    """발열 제한: 매 step 뒤 쉬어서 GPU 사용을 약 duty 비율로 맞춘다 (결과는 같다)."""
    def __init__(self, duty): self.duty = duty; self.t = time.perf_counter()
    def reset(self): self.t = time.perf_counter()
    def step(self):
        if self.duty >= 1: return
        if torch.cuda.is_available(): torch.cuda.synchronize()
        busy = time.perf_counter() - self.t; time.sleep(busy * (1 - self.duty) / max(self.duty, .05)); self.t = time.perf_counter()

# ==========================================
# 2. Parsing Dataset (측정_목록.csv + 녹음별 info CSV, 3개 스피커 녹음 1,200개)
# ==========================================
def discover(root):
    root = Path(root); recs = []
    for row in read_csv(root / '측정_목록.csv'):
        if int(row['스피커수']) != 3: continue
        p = {k: root / str(row[c]).replace('\\', '/') for k, c in [('wav', 'WAV'), ('info', '실험정보CSV'), ('angles', '각도CSV')]}
        info = {s['Speaker_Loc']: s for s in read_csv(p['info'])}
        if set(info) != set(POS) or any(s['status'] != 'OK' for s in info.values()): raise ValueError('Bad capture: ' + row['측정ID'])
        cls = [info[q]['Sound_Class'] for q in POS]
        recs.append(dict(uid=row['측정ID'], wav=str(p['wav']), angles=str(p['angles']), classes=cls,
                         fault_count=sum(c != 'Healthy' for c in cls), date=info[POS[0]]['event_time'][5:10],
                         source_files=[info[q]['Source_File'].replace('\\', '/') for q in POS],
                         rpm_intervals=[list(map(float, info[q]['Estimated_RPM_Interval'].split('~'))) for q in POS]))
    recs.sort(key=lambda r: r['uid']); return recs

# ==========================================
# 3. Feature Extraction (각도 정렬, RPM 보정, 고장 대역 지도 3장)
# ==========================================
def angle_track(path, duration):
    rows = read_csv(path); t = np.array([float(r['elapsed_sec']) for r in rows]); a = np.array([float(r['angle_deg']) for r in rows])
    keep = np.isfinite(t) & np.isfinite(a) & (t >= 0) & (t <= duration); t, a = t[keep], a[keep]
    t, u = np.unique(t, return_index=True); a = a[u]
    inc = np.r_[True, a[1:] > np.maximum.accumulate(a[:-1])]; t, a = t[inc], a[inc]
    k = (t > 0) & (t < duration) & (a > 0) & (a < 360)
    return np.r_[0.0, t[k], duration], np.r_[0.0, a[k], 360.0]

def make_rpm(r0, r1, c, dur):
    return lambda t: (lambda u: r0 + (r1 - r0) * u + c * (r1 - r0) * u * (1 - u))(np.clip(t / dur, 0, 1))

def order_spectrum(x, fs, rpm_fn, spr=512):
    """회전각 기준 재샘플링(order tracking) → (order, power dB)."""
    b, a = signal.butter(6, 10000, fs=fs); x = signal.filtfilt(b, a, x.astype(np.float64)); t = np.arange(len(x)) / fs
    rev = np.concatenate([[0], np.cumsum(rpm_fn(t[:-1]) / 60.0 / fs)]); m = int(rev[-1] * spr)
    xr = np.interp(np.arange(m) / spr, rev, x); P = np.abs(np.fft.rfft(xr * np.hanning(m))) ** 2
    return np.fft.rfftfreq(m, 1.0 / spr), 10 * np.log10(P + 1e-30)

def peak_over_bg(o, P, c, half=0.02, bg=(0.15, 0.6)):
    d = np.abs(o - c); return float(P[d <= half].max() - np.median(P[(d >= bg[0]) & (d <= bg[1])]))

def refine_rpm(audio, fs, r0, r1, dur):
    """추정 RPM(재생 구간 기반)의 시작/끝/곡률을, 정상 order 선이 가장 선명해지도록 좌표 탐색으로 보정 (라벨 미사용)."""
    def sharp(q):
        o, P = order_spectrum(audio, fs, make_rpm(r0 * (1 + q[0]), r1 * (1 + q[1]), q[2], dur))
        return float(np.mean([peak_over_bg(o, P, k) for k in CONFIG['rpm_refine_orders']]))
    p = np.zeros(3); best = sharp(p)
    for step in (0.01, 0.0025):
        improved = True
        while improved:
            improved = False
            for j in range(3):
                for d in (-step, step):
                    q = p.copy(); q[j] += d * (10 if j == 2 else 1); v = sharp(q)
                    if v > best + 1e-3: best, p, improved = v, q, True
    return p

def extract_maps(rec):
    """녹음 1개 → wind (36×58), pwm (36×68), whine (36×51). 모두 녹음 안 상대값(dB)."""
    c = CONFIG; nf, na = c['n_fft'], c['angle_bins']
    a, fs = sf.read(rec['wav'], dtype='float32'); dur = len(a) / fs; at, aa = angle_track(rec['angles'], dur)
    fr = np.fft.rfftfreq(nf, 1 / fs)
    _, t, p = signal.spectrogram(a.astype(np.float64), fs=fs, window='hann', nperseg=nf, noverlap=nf - c['hop_length'], scaling='spectrum', mode='psd')
    db = 10 * np.log10(p + 1e-30); ab = np.clip((np.interp(t, at, aa) // (360 / na)).astype(int), 0, na - 1)
    # RPM (추정 구간의 세 스피커 중앙값 → 보정)
    r0, r1 = np.median(np.array(rec['rpm_intervals']), axis=0); q = refine_rpm(a, fs, r0, r1, dur)
    rpm = make_rpm(r0 * (1 + q[0]), r1 * (1 + q[1]), q[2], dur)(t)
    # Wind: 50 Hz 칸, 5 kHz 선 제외, 양옆 직선 제거
    edg = np.arange(c['wind_band'][0], c['wind_band'][1] + 1, c['wind_bin_hz']); cen = (edg[:-1] + edg[1:]) / 2
    ok = ~((fr > c['pwm_line_exclude'][0]) & (fr < c['pwm_line_exclude'][1]))
    wind = np.stack([10 * np.log10(p[(fr >= lo) & (fr < hi) & ok].mean(0) + 1e-30) for lo, hi in zip(edg[:-1], edg[1:])])
    flk = (cen < c['wind_flanks'][0]) | (cen > c['wind_flanks'][1]); pm = np.vander(cen / 1000, 2) @ np.linalg.pinv(np.vander(cen[flk] / 1000, 2))
    # PWM: 5 kHz 주변 고해상도
    psel = np.where((fr >= c['pwm_band'][0]) & (fr < c['pwm_band'][1]))[0][:c['pwm_bins']]
    # Whine: order 16 주파수를 따라가는 좁은 띠
    off = np.arange(-c['whine_halfwidth_bins'], c['whine_halfwidth_bins'] + 1)
    wl = db[np.rint(c['whine_order'] * rpm / 60 / fr[1]).astype(int)[None, :] + off[:, None], np.arange(len(t))[None, :]]
    W = np.zeros((na, len(cen)), np.float32); Pm = np.zeros((na, len(psel)), np.float32); H = np.zeros((na, len(off)), np.float32)
    for i in range(na):
        m = ab == i
        if not m.any(): continue
        w = wind[:, m].mean(1); W[i] = w - pm @ w[flk]                    # 프레임별 dB 평균(순간 잡음에 강함)
        x = db[psel][:, m].mean(1); Pm[i] = x - np.median(x)
        h = wl[:, m].mean(1); H[i] = h - np.median(h)
    return W, Pm, H

def build_features(records):
    key = hashlib.sha256(json.dumps({k: CONFIG[k] for k in ['n_fft', 'hop_length', 'angle_bins', 'wind_band', 'wind_bin_hz', 'wind_flanks',
          'pwm_band', 'pwm_bins', 'pwm_line_exclude', 'whine_order', 'whine_halfwidth_bins', 'rpm_refine_orders']}, sort_keys=True).encode()).hexdigest()[:16]
    cache = Path(CONFIG['feature_cache_dir']) / f'features_{key}.npz'; cache.parent.mkdir(parents=True, exist_ok=True)
    if cache.exists():
        z = np.load(cache)
        if list(z['uid']) == [r['uid'] for r in records]: print('Feature cache reused:', cache.name); return [z['wind'], z['pwm'], z['whine']]
    print(f'Extracting features for {len(records)} recordings ...', flush=True)
    with ProcessPoolExecutor(CONFIG['workers']) as ex: res = list(ex.map(extract_maps, records, chunksize=8))
    maps = [np.stack([r[i] for r in res]) for i in range(3)]
    np.savez(cache, uid=np.array([r['uid'] for r in records]), wind=maps[0], pwm=maps[1], whine=maps[2]); return maps

# ==========================================
# 4. Labels — 3x3: (위치 3 × 고장 3), 3x1: (위치 3 × 고장 여부 1)
# ==========================================
def make_labels(records):
    return np.array([[[float(r['classes'][p] == c) for c in TY] for p in range(3)] for r in records], np.float32)

def labels_for(mode, Y3): return Y3 if mode == '3x3' else Y3.max(2, keepdims=True)

# ==========================================
# 5. Model Architecture
# ==========================================
class Branch(nn.Module):
    """지도 1장(각도 × 주파수)용 2D CNN. 각도 축은 원형 패딩, 주파수 축만 풀링 → (B, 64, 각도 36)."""
    def __init__(self):
        super().__init__()
        self.conv = nn.ModuleList([nn.Conv2d(a, b, 3) for a, b in ((1, 16), (16, 32), (32, 32))])
        self.norm = nn.ModuleList([nn.GroupNorm(4, b) for b in (16, 32, 32)])
    def forward(self, x):
        for i, (c, g) in enumerate(zip(self.conv, self.norm)):
            x = F.gelu(g(c(F.pad(F.pad(x, (1, 1, 0, 0)), (0, 0, 1, 1), mode='circular'))))
            if i < 2: x = F.max_pool2d(x, (1, 2))
        return torch.cat([x.mean(3), x.amax(3)], 1)

class FaultBandNet(nn.Module):
    """지도 3장 → 가지 3개 → 각도 방향 1D CNN(원형) → FC → 위치 3 × n_types logits (3x3: 9개, 3x1: 3개)."""
    def __init__(self, dropout=0.3, n_types=3):
        super().__init__(); self.nt = n_types; self.branches = nn.ModuleList([Branch() for _ in range(3)])
        self.a1 = nn.Conv1d(192, 64, 5, padding=2, padding_mode='circular'); self.a2 = nn.Conv1d(64, 64, 5, padding=2, padding_mode='circular')
        self.drop = nn.Dropout(dropout); self.out = nn.Linear(64 * CONFIG['angle_bins'], 3 * n_types)
    def forward(self, xs):
        h = torch.cat([b(x) for b, x in zip(self.branches, xs)], 1)
        h = F.gelu(self.a2(self.drop(F.gelu(self.a1(h)))))
        return self.out(self.drop(h.flatten(1))).view(-1, 3, self.nt)

# ==========================================
# 6. Evaluation Functions
# ==========================================
def recording_auc(P, Y):
    v = []
    for c in range(3):
        y = Y[:, :, c].max(1)
        if 0 < y.mean() < 1: v.append(roc_auc_score(y, P[:, :, c].max(1)))
    return float(np.mean(v))

def sector_auc(P, Y):
    """섹터(스피커) 단위: 이 스피커가 고장인가 — 검증(조기 종료)용 (3x1)."""
    return float(roc_auc_score(Y.max(2).ravel(), P.max(2).ravel()))

def sector_report(Ps, Ys, fc, date, thr):
    """Ps: (N, 3) 섹터 고장 확률, Ys: (N, 3) 0/1. 고장 개수별 정상·고장 섹터 판별, 순위, 정상 녹음 오탐."""
    pred = Ps >= thr; S = dict(sector_auc_all=float(roc_auc_score(Ys.ravel(), Ps.ravel())))
    m = (fc == 1) | (fc == 2); S['sector_auc_1_2'] = float(roc_auc_score(Ys[m].ravel(), Ps[m].ravel()))
    for q in range(4):
        k = fc == q; H = Ys[k] == 0; Fm = Ys[k] > 0; p = pred[k]
        S[f'fc{q}'] = dict(n_rec=int(k.sum()), healthy_sectors=int(H.sum()), healthy_correct=int((~p[H]).sum()),
                           fault_sectors=int(Fm.sum()), fault_correct=int(p[Fm].sum()), sector_acc=float((p == Fm).mean()),
                           all3_correct=float((p == Fm).all(1).mean()))
    k1 = np.where(fc == 1)[0]; S['fc1_fault_sector_ranked_first'] = float(np.mean([Ys[i, Ps[i].argmax()] for i in k1]))
    S['healthy_rec_flagged'] = {d: float(pred[(fc == 0) & (date == d)].any(1).mean()) for d in sorted(set(date[fc == 0]))}
    return S

def report(P, Y, fc, date, thr):
    """P, Y: (N, 위치 3, 고장 3). 고장별 검출/오탐, 종류, 위치(섹터) 지표."""
    S = {}
    for c, name in enumerate(TY):
        s = P[:, :, c].max(1); has = Y[:, :, c].max(1) > 0; pred = s >= thr; num = den = 0
        for b in sorted({(d, f) for d, f in zip(date, fc) if f in (1, 2)}):      # 같은 날짜·고장 개수 블록 안 비교
            k = (date == b[0]) & (fc == b[1])
            if has[k].any() and (~has[k]).any(): w = has[k].sum() * (~has[k]).sum(); num += roc_auc_score(has[k], s[k]) * w; den += w
        one = Y[:, :, c].sum(1) == 1; h1109 = (fc == 0) & (date == '11-09'); h1102 = (fc == 0) & (date == '11-02')
        S[name] = dict(auc=float(roc_auc_score(has, s)) if 0 < has.mean() < 1 else None, in_block_auc=float(num / den) if den else None,
                       detected=float(pred[has].mean()) if has.any() else None, n=int(has.sum()),
                       fa_healthy_1109=float(pred[h1109].mean()) if h1109.any() else None, fa_healthy_1102=float(pred[h1102].mean()) if h1102.any() else None,
                       fa_fault_rec_without=float(pred[(fc > 0) & ~has].mean()),
                       position_correct=float((P[one][:, :, c].argmax(1) == Y[one][:, :, c].argmax(1)).mean()) if one.any() else None)
    pr = P.max(1) >= thr; hs = Y.max(1) > 0; anyp = pr.any(1); det = hs.any(1); k1 = (fc == 1) & det
    S['any_fault'] = dict(recall=float(anyp[det].mean()), flagged_all_healthy=float(anyp[fc == 0].mean()) if (fc == 0).any() else None,
                          flagged_only_imbalance_modulation=float(anyp[(fc > 0) & ~det].mean()) if ((fc > 0) & ~det).any() else None)
    S['type_exact_1fault'] = float((pr[k1] == hs[k1]).all(1).mean()) if k1.any() else None
    ss = P.max(2); sy = Y.max(2); m = (fc == 1) | (fc == 2)
    S['sector'] = dict(auc_within_1_2=float(roc_auc_score(sy[m].ravel(), ss[m].ravel())),
                       fc1_fault_sector_ranked_first=float(np.mean([sy[i, ss[i].argmax()] for i in np.where(k1)[0]])) if k1.any() else None,
                       all3_exact={f'fc{q}': float(((ss[fc == q] >= thr) == (sy[fc == q] > 0)).all(1).mean()) for q in range(4) if (fc == q).any()})
    return S

# ==========================================
# 7. Train Loop
# ==========================================
def train_fold(X, Y, tr, va, seed, run_dir):
    """Y: (N, 3, n_types). n_types=3 → 3x3, 1 → 3x1. 검증 지표: 3x3은 녹음 AUC(고장 종류별), 3x1은 섹터 AUC."""
    device = torch.device(CONFIG['device']); seed_all(seed)
    mu = [float(x[tr].mean()) for x in X]; sd = [float(x[tr].std()) for x in X]                 # 정규화: train 녹음 통계만
    Z = [torch.from_numpy(((x - m) / s).astype(np.float32))[:, None] for x, m, s in zip(X, mu, sd)]; Yt = torch.from_numpy(Y)
    pos = Y[tr].reshape(len(tr), -1).sum(0)
    pw = torch.tensor(np.clip((len(tr) - pos) / np.maximum(pos, 1), 1, CONFIG['max_pos_weight']), dtype=torch.float32, device=device).view(1, 3, -1)
    model = FaultBandNet(CONFIG['dropout'], Y.shape[2]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    g = torch.Generator().manual_seed(seed); throttle = GpuThrottle(CONFIG['gpu_duty'])
    def predict(ix):
        model.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(ix), 128): b = ix[i:i + 128]; out.append(torch.sigmoid(model([q[b].to(device) for q in Z])).cpu().numpy())
        return np.concatenate(out)
    best = (-1, None, 0); history = []
    for epoch in range(1, CONFIG['epochs'] + 1):
        model.train(); perm = torch.from_numpy(tr)[torch.randperm(len(tr), generator=g)]; total = 0.0; throttle.reset()
        for i in range(0, len(perm), CONFIG['batch_size']):
            b = perm[i:i + CONFIG['batch_size']]
            loss = F.binary_cross_entropy_with_logits(model([q[b].to(device) for q in Z]), Yt[b].to(device), pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step(); total += float(loss) * len(b); throttle.step()
        val = (recording_auc if Y.shape[2] == 3 else sector_auc)(predict(va), Y[va]); history.append(dict(epoch=epoch, train_loss=total / len(tr), val_auc=val))
        if val > best[0]: best = (val, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, epoch)
        if epoch - best[2] >= CONFIG['patience']: break
    model.load_state_dict(best[1]); print(f'   best epoch {best[2]}  val AUC {best[0]:.3f}', flush=True)
    torch.save(dict(state_dict=best[1], mu=mu, sd=sd, config=dict(CONFIG), epoch=best[2]), run_dir / 'faultbandnet_best.pt')
    write_json(run_dir / 'history.json', history)
    return predict

# ==========================================
# 8. 원본 음원 비중복 반복 분할
# ==========================================
def make_repeats(records):
    """원본 파일(세 스피커 모두)을 기준으로 분할. 완전한 k-fold는 불가능(정상 원본이 최대 12번, 여러 위치에 재사용).
    매 회차: 아직 덜 시험된 녹음부터 그 원본 파일을 시험 집합 T에 넣어 파일 수가 test_files_per_repeat가 될 때까지 키움.
    시험 = 세 원본이 모두 T 안, 학습 풀 = T의 파일이 하나도 없는 녹음, 나머지는 그 회차에 쓰지 않음."""
    files = sorted({x for r in records for x in r['source_files']}); fi = {f: i for i, f in enumerate(files)}
    E = np.array([[fi[x] for x in r['source_files']] for r in records]); n = len(records)
    rng = np.random.default_rng(CONFIG['split_seed']); tested = np.zeros(n, int); reps = []
    for rep in range(CONFIG['repeats']):
        inT = np.zeros(len(files), bool); w = 1.0 / (1 + tested) ** 2
        for e in rng.choice(n, n, replace=False, p=w / w.sum()):
            if inT.sum() >= CONFIG['test_files_per_repeat']: break
            inT[E[e]] = True
        te = np.where(inT[E].all(1))[0]; tr = np.where(~inT[E].any(1))[0]
        assert not (set(E[te].ravel()) & set(E[tr].ravel()))                    # 원본 파일 겹침 0
        tested[te] += 1; reps.append(dict(repeat=rep, train_pool=tr, test=te))
    return reps

# ==========================================
# 9. Multi-Repeat Loop Start!
# ==========================================
print('Reading dataset (read-only):', CONFIG['data_root'], flush=True)
ALL = discover(CONFIG['data_root']); X_ALL = build_features(ALL)
keep = np.array([not (set(r['classes']) & set(CONFIG['exclude_classes'])) for r in ALL])
RECORDS = [r for r, k in zip(ALL, keep) if k]; X = [x[keep] for x in X_ALL]; N = len(RECORDS)
Y3 = make_labels(RECORDS); FC = np.array([r['fault_count'] for r in RECORDS]); DATE = np.array([r['date'] for r in RECORDS])
GROUP = np.array([r['source_files'][2] for r in RECORDS])                    # 검증 분할용(오른쪽 원본)
print(f'Recordings: {len(ALL)} → {N} without {CONFIG["exclude_classes"]} | by fault count {np.bincount(FC).tolist()} '
      f'| fault speakers {dict(zip(TY, Y3.sum((0, 1)).astype(int).tolist()))}')

REPEATS = make_repeats(RECORDS)
COUNT = np.zeros(N); [np.add.at(COUNT, r['test'], 1) for r in REPEATS]; TESTED = np.where(COUNT > 0)[0]
print(f'{len(REPEATS)} repeats | train pool {min(len(r["train_pool"]) for r in REPEATS)}–{max(len(r["train_pool"]) for r in REPEATS)} '
      f'| test {min(len(r["test"]) for r in REPEATS)}–{max(len(r["test"]) for r in REPEATS)} | tested at least once: {len(TESTED)}/{N}')
write_json(OUT / 'repeats.json', [dict(repeat=r['repeat'], train_pool=[RECORDS[i]['uid'] for i in r['train_pool']],
                                       test=[RECORDS[i]['uid'] for i in r['test']]) for r in REPEATS])

MODE = CONFIG['output_mode']; Y = labels_for(MODE, Y3)
if CONFIG['train']:
    if CONFIG['device'] == 'cuda' and not torch.cuda.is_available(): raise RuntimeError('CUDA unavailable.')
    S = np.zeros(Y.shape); REP_PROBS = []
    for r in REPEATS:
        tp, te = r['train_pool'], r['test']; run_dir = OUT / MODE / f"repeat{r['repeat']:02d}"; run_dir.mkdir(parents=True, exist_ok=True)
        a, b = next(GroupShuffleSplit(1, test_size=CONFIG['validation_fraction'], random_state=r['repeat']).split(tp, groups=GROUP[tp]))
        print(f"[{MODE} repeat {r['repeat']}] train {len(a)} / val {len(b)} / test {len(te)}", flush=True)
        predict = train_fold(X, Y, tp[a], tp[b], CONFIG['seed_base'] + r['repeat'], run_dir)
        p = predict(te); S[te] += p; REP_PROBS.append(p)
    PROBS = (S / np.maximum(COUNT, 1)[:, None, None]).astype(np.float32)
    np.save(OUT / f'test_probabilities_{MODE}.npy', PROBS)
    np.save(OUT / f'repeat_probabilities_{MODE}.npy', np.array(REP_PROBS, dtype=object), allow_pickle=True)
else:
    PROBS = np.load(OUT / f'test_probabilities_{MODE}.npy')
    REP_PROBS = list(np.load(OUT / f'repeat_probabilities_{MODE}.npy', allow_pickle=True))
print('Training done. Raw dataset was read only.')


In [ ]:
# ==========================================
# 1. Test Evaluation (시험된 녹음만, 회차 평균 확률)
# ==========================================
THR = CONFIG['threshold']; T = TESTED; YS = Y3.max(2)                       # YS: (N, 3) 섹터 고장 여부
PS = PROBS.max(2)                                                            # 섹터 고장 확률 (3x3: 세 고장 중 최대)
rows = []
for i in T:
    r = RECORDS[i]
    for p, pos in enumerate(POS):
        row = dict(recording_id=r['uid'], date=r['date'], fault_count=r['fault_count'], times_tested=int(COUNT[i]), position=pos, gt_class=r['classes'][p])
        if MODE == '3x3':
            row.update({f'prob_{c}': round(float(PROBS[i, p, j]), 4) for j, c in enumerate(TY)})
            row['predicted'] = next((TY[j] for j in np.argsort(-PROBS[i, p]) if PROBS[i, p, j] >= THR), 'Healthy')
        else:
            row['prob_fault'] = round(float(PROBS[i, p, 0]), 4); row['predicted'] = 'Fault' if PROBS[i, p, 0] >= THR else 'Healthy'
        rows.append(row)
write_csv(OUT / f'test_predictions_{MODE}.csv', rows)

# ==========================================
# 2. 회차별 지표 (회차마다 시험 녹음 31–41개라 흔들림이 큼)
# ==========================================
v = np.array([roc_auc_score(YS[r['test']].ravel(), p.max(2).ravel()) for r, p in zip(REPEATS, REP_PROBS) if 0 < YS[r['test']].mean() < 1])
print(f'{MODE} sector AUC per repeat: mean {v.mean():.3f} ± {v.std():.3f} (min {v.min():.3f}, max {v.max():.3f}, {len(v)} repeats)')

# ==========================================
# 3. Grand Summary
# ==========================================
SECTOR = sector_report(PS[T], YS[T], FC[T], DATE[T], THR)
TYPES = report(PROBS[T], Y3[T], FC[T], DATE[T], THR) if MODE == '3x3' else None
write_json(OUT / f'summary_{MODE}.json', dict(config=CONFIG, n_tested=int(len(T)), times_tested_mean=float(COUNT[T].mean()), sector=SECTOR, types=TYPES))
print(f'\n=== {MODE}: 섹터 1개 정상/고장 판별 (시험된 녹음 {len(T)}개, 원본 음원 비중복) ===')
fr = lambda a, b: f'{a}/{b} = {a / b:.2f}' if b else '-'
print(f"{'fault count':<12}{'rec.':>6}{'healthy sectors right':>24}{'fault sectors right':>22}{'sector acc':>12}{'all 3 right':>13}")
for q in range(4):
    s = SECTOR[f'fc{q}']
    print(f"{q:<12}{s['n_rec']:>6}{fr(s['healthy_correct'], s['healthy_sectors']):>24}{fr(s['fault_correct'], s['fault_sectors']):>22}{s['sector_acc']:>12.2f}{s['all3_correct']:>13.2f}")
print(f"sector AUC: all {SECTOR['sector_auc_all']:.3f} | within 1-/2-fault rec. {SECTOR['sector_auc_1_2']:.3f}")
print(f"1-fault rec.: fault sector ranked 1st {SECTOR['fc1_fault_sector_ranked_first']:.3f} (random 0.33)")
print('all-healthy recordings flagged:', {d: round(x, 2) for d, x in SECTOR['healthy_rec_flagged'].items()})
if MODE == '3x3':
    f2 = lambda x: '  -  ' if x is None else f'{x:.2f}'
    print('\n=== 3x3: 고장 종류별 (녹음 단위) ===')
    print(f"{'fault':<7}{'AUC':>6}{'in-block':>10}{'detected':>10}{'n':>6} | false alarms: {'healthy 11/09':>14}{'healthy 11/02':>15}{'other fault rec.':>18} | position")
    for c in TY:
        s = TYPES[c]; print(f"{c:<7}{s['auc']:>6.3f}{f2(s['in_block_auc']):>10}{s['detected']:>10.2f}{s['n']:>6} | {f2(s['fa_healthy_1109']):>28}{f2(s['fa_healthy_1102']):>15}{s['fa_fault_rec_without']:>18.2f} | {f2(s['position_correct'])}")
    print('1-fault recordings, exactly the right type:', round(TYPES['type_exact_1fault'], 3))


In [ ]:
# 결과 그림 1 (Nature style): 고장 개수별 섹터 판별 + (3x3) 고장 종류별 검출·오탐 / (3x1) 섹터 지표
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'], 'font.size': 9,
                     'axes.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8})
COL = {'Whine': '#4C72B0', 'PWM': '#DD8452', 'Wind': '#55A868'}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.4, 2.9), gridspec_kw={'width_ratios': [1.4, 1]})
xs = [(q, k) for q in range(4) for k in ('healthy', 'fault') if SECTOR[f'fc{q}'][f'{k}_sectors']]
v = [SECTOR[f'fc{q}'][f'{k}_correct'] / SECTOR[f'fc{q}'][f'{k}_sectors'] for q, k in xs]
ax1.bar(range(len(xs)), v, 0.6, color=['#9ad9c5' if k == 'healthy' else '#F4A3A3' for q, k in xs], edgecolor='k', linewidth=0.6)
for a, b in enumerate(v): ax1.text(a, b + .02, f'{b:.2f}', ha='center', fontsize=7)
ax1.set_xticks(range(len(xs))); ax1.set_xticklabels([f'{q} fault\n{k}' for q, k in xs], fontsize=7); ax1.set_ylim(0, 1.1)
ax1.set_ylabel('Sectors judged correctly'); ax1.set_title(f'{MODE}: sector level, by fault count of the recording', fontsize=9)
if MODE == '3x3':
    x = np.arange(3); w2 = 0.26
    ax2.bar(x - w2, [TYPES[c]['detected'] for c in TY], w2, color=[COL[c] for c in TY], edgecolor='k', linewidth=0.6, label='Detected')
    ax2.bar(x, [TYPES[c]['fa_fault_rec_without'] for c in TY], w2, color=[COL[c] for c in TY], alpha=0.45, edgecolor='k', linewidth=0.6, hatch='//', label='False alarm (other-fault rec.)')
    ax2.bar(x + w2, [TYPES[c]['fa_healthy_1102'] or 0 for c in TY], w2, color='white', edgecolor='k', linewidth=0.6, label='False alarm (healthy 11/02)')
    ax2.set_xticks(x); ax2.set_xticklabels(TY); ax2.legend(frameon=False, fontsize=6, loc='upper right'); ax2.set_title('Recording level, per fault type', fontsize=9)
else:
    vals = [SECTOR['sector_auc_1_2'], SECTOR['fc1_fault_sector_ranked_first'], SECTOR['fc1']['all3_correct'], SECTOR['fc2']['all3_correct']]
    ax2.bar(range(4), vals, 0.6, color='#8172B3', edgecolor='k', linewidth=0.6); ax2.axhline(1 / 3, color='gray', ls='--', lw=0.8)
    for a, b in enumerate(vals): ax2.text(a, b + .02, f'{b:.2f}', ha='center', fontsize=7)
    ax2.set_xticks(range(4)); ax2.set_xticklabels(['Sector AUC\n(1-/2-fault)', 'Fault sector\nranked 1st', 'All 3 right\n(1 fault)', 'All 3 right\n(2 faults)'], fontsize=7)
    ax2.set_title('Position (sector) level', fontsize=9)
ax2.set_ylim(0, 1.1)
plt.tight_layout(); (OUT / 'figures').mkdir(exist_ok=True)
plt.savefig(OUT / f'figures/Fig_Sector_Summary_{MODE}.pdf'); plt.savefig(OUT / f'figures/Fig_Sector_Summary_{MODE}.png', dpi=200); plt.show()


In [ ]:
# 결과 그림 2: 시험 녹음 예시 — 입력 지도 3장(각도 × 주파수)과 출력 확률(위치 × 고장)
import matplotlib.pyplot as plt
picks = [T[np.where(FC[T] == q)[0][0]] for q in range(4) if (FC[T] == q).any()]
cols = TY if MODE == '3x3' else ['fault']
fig, axes = plt.subplots(len(picks), 4, figsize=(9.5, 2.2 * len(picks)), gridspec_kw={'width_ratios': [1, 1, 1, 0.9 if MODE == '3x3' else 0.5]})
ext = [(CONFIG['wind_band'][0] / 1000, CONFIG['wind_band'][1] / 1000), (CONFIG['pwm_band'][0] / 1000, CONFIG['pwm_band'][1] / 1000), (-150, 150)]
names = ['Wind map (kHz)', 'PWM map (kHz)', 'Whine map (Hz from order 16)']
for r, i in enumerate(picks):
    for j in range(3):
        ax = axes[r, j]; ax.imshow(X[j][i], origin='lower', aspect='auto', cmap='magma', extent=[ext[j][0], ext[j][1], 0, 360], vmin=-3, vmax=12)
        ax.set_yticks([0, 90, 180, 270, 360]); ax.tick_params(labelsize=6)
        if r == 0: ax.set_title(names[j], fontsize=8)
        if j == 0: ax.set_ylabel('Mic angle (°)', fontsize=7)
    M = PROBS[i]; ax = axes[r, 3]; ax.imshow(M, cmap='Reds', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=6)
    ax.set_yticks(range(3)); ax.set_yticklabels([f'{p} ({c})' for p, c in zip(POS, RECORDS[i]['classes'])], fontsize=6)
    for a in range(3):
        for b in range(len(cols)): ax.text(b, a, f'{M[a, b]:.2f}', ha='center', va='center', fontsize=6, color='white' if M[a, b] > 0.6 else 'black')
    if r == 0: ax.set_title(f'{MODE} output (true class)', fontsize=8)
plt.tight_layout(); plt.savefig(OUT / f'figures/Fig_Test_Examples_{MODE}.pdf'); plt.savefig(OUT / f'figures/Fig_Test_Examples_{MODE}.png', dpi=200); plt.show()
